# ByT5 Akkadian Lemmatization + POS Tagging
Train a ByT5 seq2seq model mapping transliteration tokens → `lemma,POS`.


In [ ]:
!pip install transformers datasets accelerate pandas

In [ ]:
!pip install -U transformers datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 94.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 53.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 58.1 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [ ]:
!pip install conllu

## Train OA

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
from datetime import datetime

DRIVE_PROJECT_DIR = Path('/content/drive/MyDrive/FactGrid Cuneiform (AWCA)/people/Tanushri/byt5_akkadian_runs')
RUN_NAME = f"byt5_akkadian_oa_ver_3"
DRIVE_RUN_DIR = DRIVE_PROJECT_DIR / RUN_NAME
DRIVE_MODEL_DIR = DRIVE_RUN_DIR / 'model'
DRIVE_ZIP_PATH = DRIVE_PROJECT_DIR / f'{RUN_NAME}.zip'

DRIVE_MODEL_DIR.mkdir(parents=True, exist_ok=True)

print('Run folder:', DRIVE_RUN_DIR)
print('Model folder:', DRIVE_MODEL_DIR)
print('Zip path:', DRIVE_ZIP_PATH)

In [ ]:
import pandas as pd
from datasets import Dataset
from conllu import parse

# First train on AkkadianConllu_AAedit_12-2025 file
with open('/content/AkkadianConllu_AAedit_12-2025 - AkkadianConllu - AkkadianConllu_AAedit_12-2025 - AkkadianConllu.csv', 'r', encoding='utf-8') as f:
    sentences = parse(f.read())

data = []
for sent in sentences:
    for token in sent:
        form = token["form"]
        lemma = token["lemma"]

        if form and lemma:
            data.append({
                "input_text": form,
                "target_text": lemma
            })

# Convert to DataFrame
df = pd.DataFrame(data)

# Convert to HuggingFace Dataset
dataset = Dataset.from_pandas(df)

# Train on full dataset
train_dataset = dataset

In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained('google/byt5-small')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/698 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

In [ ]:
max_input_length = 64 #input char length
max_target_length = 32 #target char length

def tokenize(batch):
    model_inputs = tokenizer(
        batch['input_text'],
        truncation=True,
        max_length=max_input_length,
        padding='max_length'
    )

    labels = tokenizer(
        text_target=batch['target_text'],
        truncation=True,
        max_length=max_target_length,
        padding='max_length'
    )

    labels_ids = labels['input_ids']
    labels_ids = [
        [(l if l != tokenizer.pad_token_id else -100) for l in label]
        for label in labels_ids
    ]

    model_inputs['labels'] = labels['input_ids']
    return model_inputs

tokenized_train = train_dataset.map(tokenize, batched=True)
tokenized_dev = dev_dataset.map(tokenize, batched=True)
tokenized_train = tokenized_train.remove_columns(['input_text','target_text'])
tokenized_dev = tokenized_dev.remove_columns(['input_text','target_text'])

Map:   0%|          | 0/91733 [00:00<?, ? examples/s]

Map:   0%|          | 0/10193 [00:00<?, ? examples/s]

In [ ]:
from transformers import AutoModelForSeq2SeqLM
model = AutoModelForSeq2SeqLM.from_pretrained('google/byt5-small')

pytorch_model.bin:   0%|          | 0.00/1.20G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/172 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [ ]:
from transformers import DataCollatorForSeq2Seq
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model, label_pad_token_id=-100)

In [ ]:
from transformers import Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir=str(DRIVE_MODEL_DIR / "v3_reg"),

    eval_strategy="epoch",
    save_strategy="epoch",

    learning_rate=3e-5,   # even lower than v1
    per_device_train_batch_size=8,   # smaller batch = noisier gradients
    per_device_eval_batch_size=16,

    num_train_epochs=12,   # more epochs since learning is slower
    weight_decay=0.05,     # stronger regularization

    predict_with_generate=True,
    generation_max_length=32,

    logging_steps=50,
    save_total_limit=2,

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    report_to="none"
)

print('Checkpoints and trainer outputs will be written to:', training_args.output_dir)


Checkpoints and trainer outputs will be written to: /content/drive/MyDrive/FactGrid Cuneiform (AWCA)/people/Tanushri/byt5_akkadian_runs/byt5_akkadian_20260401_151548/model


In [ ]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

T5ForConditionalGeneration(
  (shared): Embedding(384, 1472)
  (encoder): T5Stack(
    (embed_tokens): Embedding(384, 1472)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=1472, out_features=384, bias=False)
              (k): Linear(in_features=1472, out_features=384, bias=False)
              (v): Linear(in_features=1472, out_features=384, bias=False)
              (o): Linear(in_features=384, out_features=1472, bias=False)
              (relative_attention_bias): Embedding(32, 6)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseGatedActDense(
              (wi_0): Linear(in_features=1472, out_features=3584, bias=False)
              (wi_1): Linear(in_features=1472, out_features=3584, bias=False)
              (w

In [ ]:
from transformers import Seq2SeqTrainer
trainer = Seq2SeqTrainer(
 model=model,
 args=training_args,
 train_dataset=tokenized_train,
 eval_dataset=tokenized_dev,
 data_collator=data_collator
)
trainer.train()

Epoch,Training Loss,Validation Loss
1,0.023611,0.019251
2,0.019120,0.014027
3,0.012728,0.013382
4,0.011127,0.011781
5,0.007600,0.012028
6,0.008991,0.011295
7,0.006260,0.011993
8,0.005318,0.012319
9,0.004876,0.012314
10,0.004945,0.013023


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight'].


TrainOutput(global_step=57340, training_loss=0.03294931124593058, metrics={'train_runtime': 8775.9134, 'train_samples_per_second': 104.528, 'train_steps_per_second': 6.534, 'total_flos': 1.0534956870844416e+17, 'train_loss': 0.03294931124593058, 'epoch': 10.0})

In [ ]:
trainer.save_model(str(DRIVE_MODEL_DIR))
tokenizer.save_pretrained(str(DRIVE_MODEL_DIR))

print('Final model saved to:', DRIVE_MODEL_DIR)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Final model saved to: /content/drive/MyDrive/FactGrid Cuneiform (AWCA)/people/Tanushri/byt5_akkadian_runs/byt5_akkadian_20260401_151548/model


In [ ]:
def predict(text):
    model.eval()

    inputs = tokenizer(text, return_tensors="pt")
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=5,
            num_beams=5,
            early_stopping=True
        )

    decoded = tokenizer.decode(output[0], skip_special_tokens=True)

    # Force single word
    return decoded.split()[0]

print(predict('A-sà-nu-ma'))

A-sà


In [ ]:
import shutil

archive_base = str(DRIVE_ZIP_PATH).replace('.zip', '')
created_zip = shutil.make_archive(
    base_name=archive_base,
    format='zip',
    root_dir=str(DRIVE_MODEL_DIR.parent),
    base_dir=DRIVE_MODEL_DIR.name
)

print('Zip archive created at:', created_zip)


Zip archive created at: /content/drive/MyDrive/FactGrid Cuneiform (AWCA)/people/Tanushri/byt5_akkadian_runs/byt5_akkadian_20260401_151548.zip


In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

# Replace below with model path after training and saving
model_path = "/content/drive/MyDrive/FactGrid Cuneiform (AWCA)/people/Tanushri/byt5_akkadian_runs/byt5_akkadian_20260401_151548"

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSeq2SeqLM.from_pretrained(model_path)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

ValueError: Couldn't instantiate the backend tokenizer from one of: 
(1) a `tokenizers` library serialization file, 
(2) a slow tokenizer instance to convert or 
(3) an equivalent slow tokenizer class to instantiate and convert. 
You need to have sentencepiece or tiktoken installed to convert a slow tokenizer to a fast one.

## Tests for OA

In [ ]:
import pandas as pd

lexicon_df = pd.read_csv("/content/lexicon_results - lexicon_results - lexicon_results - lexicon_results.csv")
print(lexicon_df.columns)

Index(['type', 'Female(f)', 'form', 'norm', 'lexeme', 'eBL', 'I_IV', 'A_D',
       'Alt_lex'],
      dtype='object')


In [ ]:
def predict(texts, batch_size=64):
    preds = []

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]

        inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True)
        inputs = {k: v.to(model.device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=20,
                num_beams=5,
                early_stopping=True
            )

        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)

        cleaned = [d.strip() for d in decoded]

        preds.extend(cleaned)

    return preds

input_col1 = "form"
target_col1 = "lexeme"
texts = lexicon_df[input_col1].astype(str).tolist()
#print(predict(texts[:20])) # predicting first 20 rows to see how model does
lexicon_df["byt5_pred_ver_3"] = lexicon_df[input_col1].astype(str).apply(predict)

['Ilabrat-ilī', 'Aššur', 'Aššur', 'Aššur', 'Aššur', 'šaršamš', 'šaršamš', 'a-šur-ba-ni', 'Suen-bānī', 'Damiq-bēl-š', 'Damiq-bēl-š', 'Aššur-dān', 'Aššur-dān-ma', 'Damiq-ṭāb', 'Aššur-ṣulū', 'Aššur-ṣulū', 'Aššur-eabaš', 'Aššur-emūqī-', 'Aššur-rabi', 'Aššur-idī']


In [ ]:
def normalize(x):
    return str(x).strip().lower()

lexicon_df["correct"] = (
    lexicon_df["byt5_pred_ver_3"].apply(normalize) ==
    lexicon_df[target_col1].apply(normalize)
)

accuracy = lexicon_df["correct"].mean()
print(f"Accuracy: {accuracy:.4f}")

In [ ]:
errors = lexicon_df[~lexicon_df["correct"]]
print(errors[[input_col1, target_col1, "byt5_pred_ver_3"]].head(20))

In [ ]:
oatp_df = pd.read_csv("/content/drive/MyDrive/FactGrid Cuneiform (AWCA)/people/Tanushri/OA_parses_filled_combined_AAedit_6-25.xlsx - OATP.csv")
print(oatp_df.columns)

Index(['Form', 'clean?', 'Form_clean', 'Lex_ebl', 'Lex_lower', 'Lex_corrected',
       'MA', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9', 'POS',
       'BART_Predicted_Lexeme', 'Correct (T/F)', 'Akk-Morph_foma', 'notes',
       'Unnamed: 15', 'Correct_POS_PN_GN', 'Notes', 'context', 'text',
       'Akk-Morph_foma.1', 'Gen_Tense', 'Person_Num', 'Case_encl', 'Poss',
       'Unnamed: 25'],
      dtype='object')


In [ ]:
input_col2 = "Form_clean"
target_col2 = "Lex_corrected"
bart_col = "BART_Predicted_Lexeme"
foma_col = "Akk-Morph_foma"
texts = oatp_df[input_col2].astype(str).tolist()
# print(predict(texts[:20])) # predicting first 20 rows to see how model does
oatp_df["byt5_pred_ver_3"] = oatp_df[input_col2].astype(str).apply(predict)

['Aššur', 'Aššur', 'Aššur-ṭāb', 'Aššur-idī', 'Aššur-idī', 'Aššur-mālik', 'Aššur-malik-', 'Damiq-taklāk', 'Enlil-bānī', 'Enlil-bānī', 'Enlil-bānī', 'Enlil-bānī', 'Enlil-bānī', 'Enlil-bānī-m', 'Enlil-bānī-ma', 'Enlil-bānī-ma', 'Enlil-bānī', 'Enlil-bānī', 'Enlil-bānī-m', 'Suen-nādā']


In [ ]:
def compute_acc(df, pred_col, gold_col):
    return (df[pred_col].apply(normalize) == df[gold_col].apply(normalize)).mean()

print("ByT5:", compute_acc(oatp_df, "byt5_pred_ver_3", target_col2))
print("BART:", compute_acc(oatp_df, bart_col, target_col2))
print("FOMA:", compute_acc(oatp_df, foma_col, target_col2))

In [ ]:
lexicon_df.to_csv("/content/drive/MyDrive/FactGrid Cuneiform (AWCA)/people/Tanushri/content/lexicon_results.csv", index=False)
oatp_df.to_csv("/content/drive/MyDrive/FactGrid Cuneiform (AWCA)/people/Tanushri/content/oatp_results.csv", index=False)

## Train OB

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
from datetime import datetime

DRIVE_PROJECT_DIR = Path('/content/drive/MyDrive/FactGrid Cuneiform (AWCA)/people/Tanushri/byt5_akkadian_runs')
RUN_NAME = f"byt5_akkadian_ob_ver_3"
DRIVE_RUN_DIR = DRIVE_PROJECT_DIR / RUN_NAME
DRIVE_MODEL_DIR = DRIVE_RUN_DIR / 'model'
DRIVE_ZIP_PATH = DRIVE_PROJECT_DIR / f'{RUN_NAME}.zip'

DRIVE_MODEL_DIR.mkdir(parents=True, exist_ok=True)

print('Run folder:', DRIVE_RUN_DIR)
print('Model folder:', DRIVE_MODEL_DIR)
print('Zip path:', DRIVE_ZIP_PATH)

In [ ]:
import pandas as pd
from datasets import Dataset
from conllu import parse

# Train on lemmatization_train_no_ids.csv file
with open('/content/lemmatization_train_no_ids.csv', 'r', encoding='utf-8') as f:
    sentences = parse(f.read())

data = []
for sent in sentences:
    for token in sent:
        form = token["form"]
        lemma = token["lemma"]

        if form and lemma:
            data.append({
                "input_text": form,
                "target_text": lemma
            })

# Convert to DataFrame
df = pd.DataFrame(data)

# Convert to HuggingFace Dataset
dataset = Dataset.from_pandas(df)

# Train-test split
dataset = dataset.train_test_split(test_size=0.3)
train_dataset = dataset['train']
dev_dataset = dataset['test']

In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained('google/byt5-small')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/698 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

In [ ]:
max_input_length = 64 #input char length
max_target_length = 32 #target char length

def tokenize(batch):
    model_inputs = tokenizer(
        batch['input_text'],
        truncation=True,
        max_length=max_input_length,
        padding='max_length'
    )

    labels = tokenizer(
        text_target=batch['target_text'],
        truncation=True,
        max_length=max_target_length,
        padding='max_length'
    )

    labels_ids = labels['input_ids']
    labels_ids = [
        [(l if l != tokenizer.pad_token_id else -100) for l in label]
        for label in labels_ids
    ]

    model_inputs['labels'] = labels['input_ids']
    return model_inputs

tokenized_train = train_dataset.map(tokenize, batched=True)
tokenized_dev = dev_dataset.map(tokenize, batched=True)
tokenized_train = tokenized_train.remove_columns(['input_text','target_text'])
tokenized_dev = tokenized_dev.remove_columns(['input_text','target_text'])

Map:   0%|          | 0/91733 [00:00<?, ? examples/s]

Map:   0%|          | 0/10193 [00:00<?, ? examples/s]

In [ ]:
from transformers import AutoModelForSeq2SeqLM
model = AutoModelForSeq2SeqLM.from_pretrained('google/byt5-small')

pytorch_model.bin:   0%|          | 0.00/1.20G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/172 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [ ]:
from transformers import DataCollatorForSeq2Seq
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model, label_pad_token_id=-100)

In [ ]:
from transformers import Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir=str(DRIVE_MODEL_DIR / "v3_reg"),

    eval_strategy="epoch",
    save_strategy="epoch",

    learning_rate=3e-5,   # even lower than v1
    per_device_train_batch_size=8,   # smaller batch = noisier gradients
    per_device_eval_batch_size=16,

    num_train_epochs=12,   # more epochs since learning is slower
    weight_decay=0.05,     # stronger regularization

    predict_with_generate=True,
    generation_max_length=32,

    logging_steps=50,
    save_total_limit=2,

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    report_to="none"
)

print('Checkpoints and trainer outputs will be written to:', training_args.output_dir)


Checkpoints and trainer outputs will be written to: /content/drive/MyDrive/FactGrid Cuneiform (AWCA)/people/Tanushri/byt5_akkadian_runs/byt5_akkadian_20260401_151548/model


In [ ]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

T5ForConditionalGeneration(
  (shared): Embedding(384, 1472)
  (encoder): T5Stack(
    (embed_tokens): Embedding(384, 1472)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=1472, out_features=384, bias=False)
              (k): Linear(in_features=1472, out_features=384, bias=False)
              (v): Linear(in_features=1472, out_features=384, bias=False)
              (o): Linear(in_features=384, out_features=1472, bias=False)
              (relative_attention_bias): Embedding(32, 6)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseGatedActDense(
              (wi_0): Linear(in_features=1472, out_features=3584, bias=False)
              (wi_1): Linear(in_features=1472, out_features=3584, bias=False)
              (w

In [ ]:
from transformers import Seq2SeqTrainer
trainer = Seq2SeqTrainer(
 model=model,
 args=training_args,
 train_dataset=tokenized_train,
 eval_dataset=tokenized_dev,
 data_collator=data_collator
)
trainer.train()

Epoch,Training Loss,Validation Loss
1,0.023611,0.019251
2,0.019120,0.014027
3,0.012728,0.013382
4,0.011127,0.011781
5,0.007600,0.012028
6,0.008991,0.011295
7,0.006260,0.011993
8,0.005318,0.012319
9,0.004876,0.012314
10,0.004945,0.013023


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight'].


TrainOutput(global_step=57340, training_loss=0.03294931124593058, metrics={'train_runtime': 8775.9134, 'train_samples_per_second': 104.528, 'train_steps_per_second': 6.534, 'total_flos': 1.0534956870844416e+17, 'train_loss': 0.03294931124593058, 'epoch': 10.0})

In [ ]:
trainer.save_model(str(DRIVE_MODEL_DIR))
tokenizer.save_pretrained(str(DRIVE_MODEL_DIR))

print('Final model saved to:', DRIVE_MODEL_DIR)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Final model saved to: /content/drive/MyDrive/FactGrid Cuneiform (AWCA)/people/Tanushri/byt5_akkadian_runs/byt5_akkadian_20260401_151548/model


In [ ]:
def predict(text):
    model.eval()

    inputs = tokenizer(text, return_tensors="pt")
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=5,
            num_beams=5,
            early_stopping=True
        )

    decoded = tokenizer.decode(output[0], skip_special_tokens=True)

    # Force single word
    return decoded.split()[0]

print(predict('A-sà-nu-ma'))

A-sà


In [ ]:
import shutil

archive_base = str(DRIVE_ZIP_PATH).replace('.zip', '')
created_zip = shutil.make_archive(
    base_name=archive_base,
    format='zip',
    root_dir=str(DRIVE_MODEL_DIR.parent),
    base_dir=DRIVE_MODEL_DIR.name
)

print('Zip archive created at:', created_zip)


Zip archive created at: /content/drive/MyDrive/FactGrid Cuneiform (AWCA)/people/Tanushri/byt5_akkadian_runs/byt5_akkadian_20260401_151548.zip


In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

# Replace below with model path after training and saving
model_path = "/content/drive/MyDrive/FactGrid Cuneiform (AWCA)/people/Tanushri/byt5_akkadian_runs/byt5_akkadian_20260401_151548"

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSeq2SeqLM.from_pretrained(model_path)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

ValueError: Couldn't instantiate the backend tokenizer from one of: 
(1) a `tokenizers` library serialization file, 
(2) a slow tokenizer instance to convert or 
(3) an equivalent slow tokenizer class to instantiate and convert. 
You need to have sentencepiece or tiktoken installed to convert a slow tokenizer to a fast one.

## Tests for OB